# 04 - Modélisation et Entraînement
## BTC Multivariate Forecast
### Étudiant B - Phase 2

In [1]:
# Import des librairies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import des modèles
import sys
sys.path.append('..')
from src.models.arima import ARIMAModel
from src.models.var import VARModel
from src.models.regression import RegressionModel
from src.models.random_forest import RandomForestModel
from src.evaluation.metrics import compute_metrics, print_metrics

# Configuration
%matplotlib inline

In [2]:
# Chargement des données préparées
X_train = pd.read_csv('../data/processed/X_train.csv', index_col=0, parse_dates=True)
X_test = pd.read_csv('../data/processed/X_test.csv', index_col=0, parse_dates=True)
y_train = pd.read_csv('../data/processed/y_train.csv', index_col=0, parse_dates=True).squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv', index_col=0, parse_dates=True).squeeze()

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

X_train: (1000, 20)
X_test: (251, 20)
y_train: (1000,)
y_test: (251,)


## 1. Modèle ARIMA (Benchmark univarié)

In [3]:
# Test de différents ordres ARIMA
arima_orders = [(1,0,0), (2,0,0), (0,0,1), (1,0,1), (2,0,1)]
arima_results = []

for order in arima_orders:
    print(f"\nTest ARIMA{order}")
    model = ARIMAModel(order=order)
    model.fit(X_train, y_train)
    
    # Prédiction sur test
    y_pred = model.predict(X_test)
    
    # Métriques
    metrics = compute_metrics(y_test.values, y_pred)
    arima_results.append({'order': order, **metrics})

# Meilleur modèle
arima_df = pd.DataFrame(arima_results).sort_values('RMSE')
arima_df


Test ARIMA(1, 0, 0)
✓ ARIMA entraîné avec succès

Test ARIMA(2, 0, 0)
✓ ARIMA entraîné avec succès

Test ARIMA(0, 0, 1)
✓ ARIMA entraîné avec succès

Test ARIMA(1, 0, 1)
✓ ARIMA entraîné avec succès

Test ARIMA(2, 0, 1)
✓ ARIMA entraîné avec succès


,order,MAE,RMSE,MAPE,Direction_Accuracy,R2,n_obs
4,"(2, 0, 1)",0.025871,0.035068,108.486033,17.2,0.000362,251
1,"(2, 0, 0)",0.025871,0.035073,107.634306,5.6,0.000092,251
0,"(1, 0, 0)",0.025884,0.035079,108.453878,4.0,-0.000259,251
3,"(1, 0, 1)",0.025880,0.035079,107.860262,4.8,-0.000282,251
2,"(0, 0, 1)",0.025884,0.035084,107.863721,0.4,-0.000524,251


In [4]:
# Modèle ARIMA final
best_order = arima_df.iloc[0]['order']
arima_model = ARIMAModel(order=best_order, name=f"ARIMA{best_order}")
arima_model.fit(X_train, y_train)
y_pred_arima = arima_model.predict(X_test)

metrics_arima = compute_metrics(y_test.values, y_pred_arima)
print_metrics(metrics_arima, arima_model.name)

✓ ARIMA(2, 0, 1) entraîné avec succès

MÉTRIQUES D'ÉVALUATION - ARIMA(2, 0, 1)
MAE  : 0.025871
RMSE : 0.035068
MAPE : 108.49%
DIR  : 17.20%
R²   : 0.0004
N    : 251


## 2. Modèle VAR (Multivarié)

In [5]:
# Test différents maxlags
var_model = VARModel(maxlags=10)
var_model.fit(X_train, y_train)

# Prédiction
y_pred_var = var_model.predict(X_test)
metrics_var = compute_metrics(y_test.values, y_pred_var)
print_metrics(metrics_var, var_model.name)

# Afficher les lags sélectionnés
print(f"\nLags sélectionnés: {var_model.selected_lags}")

✓ VAR entraîné avec 1 lags

MÉTRIQUES D'ÉVALUATION - VAR
MAE  : 0.025945
RMSE : 0.035152
MAPE : 112.49%
DIR  : 29.60%
R²   : -0.0044
N    : 251

Lags sélectionnés: 1


## 3. Modèles de Régression

In [6]:
# Test Ridge avec différents alpha
alphas = [0.1, 1.0, 10.0, 100.0]
ridge_results = []

for alpha in alphas:
    model = RegressionModel(model_type='ridge', alpha=alpha)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    metrics = compute_metrics(y_test.values, y_pred)
    ridge_results.append({'alpha': alpha, **metrics})

ridge_df = pd.DataFrame(ridge_results).sort_values('RMSE')
ridge_df

✓ RidgeRegression entraîné avec succès
  Top 5 coefficients:
    ETH_Close_log_return_lag_4: -0.0069
    DXY_Close_log_return_lag_2: 0.0050
    SP500_Close_log_return_lag_4: 0.0036
    ETH_Close_log_return_lag_1: 0.0018
    SP500_Close_log_return_lag_2: 0.0016
✓ RidgeRegression entraîné avec succès
  Top 5 coefficients:
    ETH_Close_log_return_lag_4: -0.0069
    DXY_Close_log_return_lag_2: 0.0050
    SP500_Close_log_return_lag_4: 0.0036
    ETH_Close_log_return_lag_1: 0.0017
    SP500_Close_log_return_lag_2: 0.0015
✓ RidgeRegression entraîné avec succès
  Top 5 coefficients:
    ETH_Close_log_return_lag_4: -0.0067
    DXY_Close_log_return_lag_2: 0.0049
    SP500_Close_log_return_lag_4: 0.0035
    ETH_Close_log_return_lag_1: 0.0017
    SP500_Close_log_return_lag_2: 0.0015
✓ RidgeRegression entraîné avec succès
  Top 5 coefficients:
    ETH_Close_log_return_lag_4: -0.0055
    DXY_Close_log_return_lag_2: 0.0044
    SP500_Close_log_return_lag_4: 0.0030
    ETH_Close_log_return_lag_1: 0.00

,alpha,MAE,RMSE,MAPE,Direction_Accuracy,R2,n_obs
3,100.0,0.026397,0.035326,145.799674,52.0,-0.014388,251
2,10.0,0.026494,0.035413,152.573966,52.4,-0.019423,251
1,1.0,0.026508,0.035425,153.469445,51.6,-0.020115,251
0,0.1,0.026510,0.035427,153.562161,51.6,-0.020186,251


In [7]:
# Meilleur modèle Ridge
best_alpha = ridge_df.iloc[0]['alpha']
ridge_model = RegressionModel(model_type='ridge', alpha=best_alpha, 
                             name=f"Ridge(alpha={best_alpha})")
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

metrics_ridge = compute_metrics(y_test.values, y_pred_ridge)
print_metrics(metrics_ridge, ridge_model.name)

# Feature importance
importance = ridge_model.get_feature_importance(X_train.columns)
print("\nTop 10 features importantes:")
importance.head(10)

✓ Ridge(alpha=100.0) entraîné avec succès
  Top 5 coefficients:
    ETH_Close_log_return_lag_4: -0.0055
    DXY_Close_log_return_lag_2: 0.0044
    SP500_Close_log_return_lag_4: 0.0030
    ETH_Close_log_return_lag_1: 0.0014
    DXY_Close_log_return_lag_1: -0.0014

MÉTRIQUES D'ÉVALUATION - Ridge(alpha=100.0)
MAE  : 0.026397
RMSE : 0.035326
MAPE : 145.80%
DIR  : 52.00%
R²   : -0.0144
N    : 251

Top 10 features importantes:


,feature,coefficient,abs_coefficient
3,ETH_Close_log_return_lag_4,-0.005458,0.005458
16,DXY_Close_log_return_lag_2,0.004391,0.004391
13,SP500_Close_log_return_lag_4,0.003007,0.003007
0,ETH_Close_log_return_lag_1,0.001419,0.001419
15,DXY_Close_log_return_lag_1,-0.001354,0.001354
11,SP500_Close_log_return_lag_2,0.001259,0.001259
4,ETH_Close_log_return_lag_5,-0.001233,0.001233
2,ETH_Close_log_return_lag_3,0.001152,0.001152
6,BNB_Close_log_return_lag_2,0.001144,0.001144
17,DXY_Close_log_return_lag_3,-0.001041,0.001041


In [8]:
# Test Lasso
lasso_model = RegressionModel(model_type='lasso', alpha=0.01)
lasso_model.fit(X_train, y_train)
y_pred_lasso = lasso_model.predict(X_test)

metrics_lasso = compute_metrics(y_test.values, y_pred_lasso)
print_metrics(metrics_lasso, lasso_model.name)

# Lasso fait de la sélection de variables
importance_lasso = lasso_model.get_feature_importance(X_train.columns)
n_selected = (importance_lasso['coefficient'] != 0).sum()
print(f"\nVariables sélectionnées par Lasso: {n_selected}/{len(X_train.columns)}")
importance_lasso[importance_lasso['coefficient'] != 0].head(10)

✓ LassoRegression entraîné avec succès
  Top 5 coefficients:
    ETH_Close_log_return_lag_1: 0.0000
    ETH_Close_log_return_lag_2: 0.0000
    ETH_Close_log_return_lag_3: 0.0000
    ETH_Close_log_return_lag_4: -0.0000
    ETH_Close_log_return_lag_5: -0.0000

MÉTRIQUES D'ÉVALUATION - LassoRegression
MAE  : 0.025896
RMSE : 0.035095
MAPE : 108.49%
DIR  : 0.00%
R²   : -0.0012
N    : 251

Variables sélectionnées par Lasso: 0/20


,feature,coefficient,abs_coefficient


## 4. Random Forest

In [9]:
# Random Forest basique
rf_model = RandomForestModel(n_estimators=100, max_depth=10)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

metrics_rf = compute_metrics(y_test.values, y_pred_rf)
print_metrics(metrics_rf, rf_model.name)

✓ RandomForest entraîné avec succès
  100 arbres
  Top 5 features importantes:
    DXY_Close_log_return_lag_1: 0.1448
    ETH_Close_log_return_lag_4: 0.0833
    ETH_Close_log_return_lag_1: 0.0722
    DXY_Close_log_return_lag_3: 0.0714
    ETH_Close_log_return_lag_3: 0.0627

MÉTRIQUES D'ÉVALUATION - RandomForest
MAE  : 0.026562
RMSE : 0.036570
MAPE : 130.50%
DIR  : 54.40%
R²   : -0.0871
N    : 251


In [10]:
# Optimisation des hyperparamètres
rf_optimized = RandomForestModel()
best_params = rf_optimized.optimize_params(X_train, y_train)

# Ré-entraîner avec les meilleurs paramètres
rf_optimized.fit(X_train, y_train)
y_pred_rf_opt = rf_optimized.predict(X_test)

metrics_rf_opt = compute_metrics(y_test.values, y_pred_rf_opt)
print_metrics(metrics_rf_opt, "RandomForest_Optimized")

# Feature importance
rf_importance = rf_optimized.get_feature_importance(X_train.columns)
print("\nTop 10 features Random Forest:")
rf_importance.head(10)

Optimisation des hyperparamètres...
Meilleurs paramètres: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2}
Meilleur score: 0.8337
✓ RandomForest entraîné avec succès
  200 arbres
  Top 5 features importantes:
    DXY_Close_log_return_lag_1: 0.1065
    ETH_Close_log_return_lag_4: 0.0626
    DXY_Close_log_return_lag_3: 0.0626
    ETH_Close_log_return_lag_1: 0.0621
    ETH_Close_log_return_lag_3: 0.0587

MÉTRIQUES D'ÉVALUATION - RandomForest_Optimized
MAE  : 0.026870
RMSE : 0.036915
MAPE : 139.40%
DIR  : 52.40%
R²   : -0.1077
N    : 251

Top 10 features Random Forest:


,feature,importance
15,DXY_Close_log_return_lag_1,0.106458
3,ETH_Close_log_return_lag_4,0.062584
17,DXY_Close_log_return_lag_3,0.062564
0,ETH_Close_log_return_lag_1,0.062100
2,ETH_Close_log_return_lag_3,0.058655
11,SP500_Close_log_return_lag_2,0.054792
14,SP500_Close_log_return_lag_5,0.050919
10,SP500_Close_log_return_lag_1,0.049027
8,BNB_Close_log_return_lag_4,0.048674
16,DXY_Close_log_return_lag_2,0.046591


## 5. Comparaison rapide

In [11]:
# Compilation des résultats
all_metrics = {
    'ARIMA': metrics_arima,
    'VAR': metrics_var,
    'Ridge': metrics_ridge,
    'Lasso': metrics_lasso,
    'RandomForest': metrics_rf,
    'RandomForest_Opt': metrics_rf_opt
}

from src.evaluation.metrics import compare_models
comparison = compare_models(all_metrics)
comparison

,Modèle,MAE,RMSE,MAPE (%),Direction (%),R²
0,ARIMA,0.025871,0.035068,108.49,17.2,0.0004
3,Lasso,0.025896,0.035095,108.49,0.0,-0.0012
1,VAR,0.025945,0.035152,112.49,29.6,-0.0044
2,Ridge,0.026397,0.035326,145.80,52.0,-0.0144
4,RandomForest,0.026562,0.036570,130.50,54.4,-0.0871
5,RandomForest_Opt,0.026870,0.036915,139.40,52.4,-0.1077


In [12]:
# Sauvegarde des modèles pour évaluation
import joblib
joblib.dump(arima_model, '../experiments/arima_results/arima_model.pkl')
joblib.dump(var_model, '../experiments/var_results/var_model.pkl')
joblib.dump(ridge_model, '../experiments/regression_results/ridge_model.pkl')
joblib.dump(rf_optimized, '../experiments/rf_results/rf_model.pkl')

print("✓ Modèles sauvegardés")

✓ Modèles sauvegardés
